In [7]:

%pip install pandas rich ipywidgets



   ---------------------------------------- 0.0/2.2 MB ? eta -:--:--
   --------------------------------- ------ 1.8/2.2 MB 9.1 MB/s eta 0:00:01
   ---------------------------------------- 2.2/2.2 MB 8.8 MB/s eta 0:00:00

   ------------- -------------------------- 1/3 [jupyterlab_widgets]
   ------------- -------------------------- 1/3 [jupyterlab_widgets]
   -------------------------- ------------- 2/3 [ipywidgets]
   ---------------------------------------- 3/3 [ipywidgets]

Note: you may need to restart the kernel to use updated packages.


In [ ]:
print(df['cosine_weighted'].iloc[0])
print(type(df['cosine_weighted'].iloc[0]))


[0.34644711940374345, 0.0, 0.0]
<class 'str'>


In [ ]:
import pandas as pd
import ipywidgets as widgets
from IPython.display import display, clear_output

# === Load and Prepare Data ===
file_path = "Corex_topicdata_slavery_v4/relabeled_output.csv"
df = pd.read_csv(file_path)

def safe_parse_list(x):
    if isinstance(x, list):
        return x
    if isinstance(x, float) and not pd.isna(x):
        return [x]
    if isinstance(x, str):
        if x.startswith('['):
            try:
                return eval(x)
            except:
                return []
        elif '|' in x:
            return [t.strip() for t in x.split('|') if t.strip()]
        elif x.strip():
            try:
                return [float(x.strip())]
            except:
                return [x.strip()]
    return []

df['topic'] = df['topic'].apply(safe_parse_list)
df['topic_assigned'] = df['topic_assigned'].apply(safe_parse_list)
for col in ['cosine', 'cosine_weighted']:
    df[col] = df[col].apply(safe_parse_list)

all_topics = sorted(set(t for row in df['topic'] if isinstance(row, list) for t in row))

# === Interactive Labeling Tool ===
class TopicLabeler:
    def __init__(self, df, topic_names, sort_by='cosine_weighted', sort_ascending=False):
        self.original_df = df.copy()
        self.topic_names = sorted(set(topic_names))
        self.sort_by = sort_by
        self.sort_ascending = sort_ascending
        self.df = self.sort_dataframe(df)
        self.index = 0
        self.build_ui()

    def sort_dataframe(self, df):
        ascending = self.sort_ascending
        if isinstance(df[self.sort_by].iloc[0], list):
            df = df.copy()
            df['sort_score'] = df[self.sort_by].apply(lambda x: max(x) if x else 0)
            return df.sort_values(by='sort_score', ascending=ascending).reset_index(drop=True)
        return df.sort_values(by=self.sort_by, ascending=ascending).reset_index(drop=True)

    def build_ui(self):
        clear_output(wait=True)
        self.out = widgets.Output(layout={'height': '300px', 'overflow': 'auto'})
        self.label_box = widgets.SelectMultiple(
            options=self.topic_names,
            description="Labels",
            layout=widgets.Layout(width='auto', height='140px')
        )
        self.cosine_slider = widgets.FloatSlider(
            value=1.0, min=0.0, max=1.0, step=0.01,
            description='Manual Cosine:'
        )
        self.sort_dropdown = widgets.Dropdown(
            options=['cosine', 'cosine_weighted'],
            description='Sort by:',
            value=self.sort_by
        )
        self.sort_order_dropdown = widgets.Dropdown(
            options=[('Highest First', False), ('Lowest First', True)],
            value=self.sort_ascending,
            description='Order:'
        )
        self.sort_dropdown.observe(self.change_sort, names='value')
        self.sort_order_dropdown.observe(self.change_sort, names='value')

        self.approve_button = widgets.Button(description="✅ Add / Update", button_style='success')
        self.remove_selected_button = widgets.Button(description="➖ Remove Selected", button_style='warning')
        self.clear_all_button = widgets.Button(description="❌ Clear All Labels", button_style='danger')
        self.remove_row_button = widgets.Button(description="🗑️ Remove Row", button_style='danger')
        self.skip_button = widgets.Button(description="⏭️ Skip", button_style='')
        self.next_button = widgets.Button(description="➡️ Next", button_style='info')
        self.prev_button = widgets.Button(description="⬅️ Prev", button_style='info')
        self.save_button = widgets.Button(description="💾 Save to CSV", button_style='primary')

        self.approve_button.on_click(self.approve_labels)
        self.remove_selected_button.on_click(self.remove_selected_labels)
        self.clear_all_button.on_click(self.remove_labels)
        self.remove_row_button.on_click(self.remove_row)
        self.next_button.on_click(self.next_row)
        self.prev_button.on_click(self.prev_row)
        self.save_button.on_click(self.save_to_csv)
        self.skip_button.on_click(self.skip_row)

        control_box = widgets.VBox([
            widgets.HBox([self.sort_dropdown, self.sort_order_dropdown]),
            widgets.HBox([self.prev_button, self.next_button, self.skip_button, self.save_button]),
            self.label_box,
            self.cosine_slider,
            widgets.HBox([
                self.approve_button,
                self.remove_selected_button,
                self.clear_all_button,
                self.remove_row_button
            ]),
            self.out
        ])
        display(control_box)
        self.show_current_row()

    def show_current_row(self):
        self.out.clear_output(wait=True)
        if 0 <= self.index < len(self.df):
            row = self.df.loc[self.index]
            with self.out:
                print(f"Chunk {self.index+1}/{len(self.df)}")
                print(f"Filename: {row['filename']} ({row.get('year', '')}) - {row.get('document_type', '')}")
                print(f"\nText:\n{'-'*80}\n{row['chunk_text']}\n{'-'*80}")
                print(f"\nCurrent Labels: {row['topic']}")
                # Cosines per label
                cosine_val = row['cosine']
                if isinstance(cosine_val, list):
                    print(f"Cosines: {cosine_val}")
                else:
                    print(f"Cosine: {cosine_val}")
                cosine_weighted_val = row['cosine_weighted']
                if isinstance(cosine_weighted_val, list):
                    print(f"Weighted Cosines: {cosine_weighted_val}")
                else:
                    print(f"Weighted Cosine: {cosine_weighted_val}")

    def approve_labels(self, b):
        labels_to_add = list(self.label_box.value)
        cosine = self.cosine_slider.value

        current_labels = self.df.at[self.index, 'topic']
        current_cos = self.df.at[self.index, 'cosine']
        current_weighted = self.df.at[self.index, 'cosine_weighted']

        for label in labels_to_add:
            if label in current_labels:
                i = current_labels.index(label)
                current_cos[i] = cosine
                current_weighted[i] = cosine
            else:
                current_labels.append(label)
                current_cos.append(cosine)
                current_weighted.append(cosine)

        self.df.at[self.index, 'topic'] = current_labels
        self.df.at[self.index, 'cosine'] = current_cos
        self.df.at[self.index, 'cosine_weighted'] = current_weighted
        self.show_current_row()

    def remove_selected_labels(self, b):
        labels_to_remove = list(self.label_box.value)
        current_labels = self.df.at[self.index, 'topic']
        current_cos = self.df.at[self.index, 'cosine']
        current_weighted = self.df.at[self.index, 'cosine_weighted']

        indices = [i for i, t in enumerate(current_labels) if t in labels_to_remove]
        for i in sorted(indices, reverse=True):
            if i < len(current_labels):
                current_labels.pop(i)
            if i < len(current_cos):
                current_cos.pop(i)
            if i < len(current_weighted):
                current_weighted.pop(i)

        self.df.at[self.index, 'topic'] = current_labels
        self.df.at[self.index, 'cosine'] = current_cos
        self.df.at[self.index, 'cosine_weighted'] = current_weighted
        self.show_current_row()

    def remove_labels(self, b):
        self.df.at[self.index, 'topic'] = []
        self.df.at[self.index, 'cosine'] = []
        self.df.at[self.index, 'cosine_weighted'] = []
        self.show_current_row()

    def remove_row(self, b):
        self.df = self.df.drop(self.index).reset_index(drop=True)
        if self.index >= len(self.df):
            self.index = len(self.df) - 1
        self.show_current_row()

    def skip_row(self, b):
        self.index += 1
        if self.index >= len(self.df):
            self.index = len(self.df) - 1
        self.show_current_row()

    def next_row(self, b):
        if self.index < len(self.df) - 1:
            self.index += 1
            self.show_current_row()

    def prev_row(self, b):
        if self.index > 0:
            self.index -= 1
            self.show_current_row()

    def save_to_csv(self, b):
        self.df.to_csv("relabeled_output.csv", index=False)
        with self.out:
            print("✅ Data saved to relabeled_output.csv")

    def change_sort(self, change):
        # Handle both sort column and order dropdowns
        if change['owner'] == self.sort_dropdown:
            self.sort_by = change['new']
        elif change['owner'] == self.sort_order_dropdown:
            self.sort_ascending = change['new']
        self.df = self.sort_dataframe(self.original_df)
        self.index = 0
        self.show_current_row()

# === Launch the Tool ===
labeler = TopicLabeler(df, all_topics)


FileNotFoundError: [Errno 2] No such file or directory: 'Corex_topicdata_slavery_v4/relabeled_output.csv'

: 